In [300]:
from pathlib import Path

input_dir = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\input")

In [301]:
gml_content = """<?xml version="1.0" encoding="ISO-8859-1"?>
<OpenGeoSysGLI>
    <name>simple_geometry</name>
    <points>
    </points>
    <polylines>
    </polylines>
</OpenGeoSysGLI>
"""

gml_path = input_dir / "simple_geometry.gml"
gml_path.write_text(gml_content)

print("Geometry file created:", gml_path)

Geometry file created: E:\ADATA\LITHIUM\OGS\OGS_files\input\simple_geometry.gml


In [302]:
# --------------------------------------------------
# 4. Create a complete minimal OGS project file
# --------------------------------------------------

prj_content = """<?xml version="1.0" encoding="ISO-8859-1"?>
<OpenGeoSysProject>

    <geometry>simple_geometry.gml</geometry>

    <meshes>
        <mesh>mesh6.vtu</mesh>
    
        <mesh>Layer5_bottom.vtu</mesh>
        <mesh>Layer4_caprock2.vtu</mesh>
        <mesh>Layer3_Reservoir.vtu</mesh>
        <mesh>Layer2_caprock1.vtu</mesh>
        <mesh>Layer1_Top.vtu</mesh>
    
        <mesh>Well1_bottom.vtu</mesh>
        <mesh>Well2_bottom.vtu</mesh>
    </meshes>


    <processes>
        <process>
            <name>LiquidFlow</name>
            <type>LIQUID_FLOW</type>
            <integration_order>2</integration_order>
            <specific_body_force>0 0 0</specific_body_force>
            <process_variables>
                <process_variable>pressure</process_variable>
            </process_variables>
        </process>
    </processes>


    <media>
        <medium id="0">
            <phases>
                <phase>
                    <type>AqueousLiquid</type>
                    <properties>
                        <property>
                            <name>viscosity</name>
                            <type>Constant</type>
                            <value>1e-3</value>
                        </property>
                        <property>
                            <name>density</name>
                            <type>Constant</type>
                            <value>1000</value>
                        </property>
                    </properties>
                </phase>
            </phases>
            
            <properties>
                <property>
                    <name>reference_temperature</name>
                    <type>Constant</type>
                    <value>293.15</value>
                </property>
                <property>
                    <name>permeability</name>
                    <type>Constant</type>
                    <value>1e-8</value>
                </property>
                <property>
                    <name>porosity</name>
                    <type>Constant</type>
                    <value>0.2</value>
                </property>
                <property>
                    <name>storage</name>
                    <type>Constant</type>
                    <value>1e-10</value>
                </property>
            </properties>
        </medium>
    </media>

    <time_loop>
        <processes>
            <process ref="LiquidFlow">
                <nonlinear_solver>basic_picard</nonlinear_solver>
                <time_discretization>
                    <type>BackwardEuler</type>
                </time_discretization>
                <convergence_criterion>
                    <type>DeltaX</type>
                    <norm_type>NORM2</norm_type>
                    <abstol>1e-12</abstol>
                </convergence_criterion>
                <time_stepping>
                    <type>FixedTimeStepping</type>
                    <t_initial>0</t_initial>
                    <t_end>10</t_end>
                    <timesteps>
                        <pair>
                            <repeat>10</repeat>
                            <delta_t>1</delta_t>
                        </pair>
                    </timesteps>
                </time_stepping>
            </process>
        </processes>

        <output>
            <type>VTK</type>
            <prefix>result</prefix>
            <timesteps>
                <pair>
                    <repeat>10</repeat>
                    <each_steps>1</each_steps>
                </pair>
            </timesteps>
            <variables>
                <variable>pressure</variable>
            </variables>
        </output>
    </time_loop>

    <parameters>
        <parameter>
            <name>pressure_ic</name>
            <type>Constant</type>
            <value>1e6</value>
        </parameter>
        <parameter>
            <name>p_injection</name>
            <type>Constant</type>
            <value>1e6</value>
        </parameter>
        <parameter>
            <name>p_production</name>
            <type>Constant</type>
            <value>0</value>
        </parameter>
        <parameter>
            <name>q_injection</name>
            <type>Constant</type>
            <value>1e-1</value>
        </parameter>
    </parameters>

    <process_variables>
        <process_variable>
            <name>pressure</name>
            <components>1</components>
            <order>1</order>
            <initial_condition>pressure_ic</initial_condition>
            
            <boundary_conditions>
                <boundary_condition>
                    <type>Neumann</type>
                    <mesh>well1_completion</mesh>
                    <parameter>q_injection</parameter>
                </boundary_condition>
                <boundary_condition>
                    <type>Dirichlet</type>
                    <mesh>well2_completion</mesh>
                    <parameter>p_production</parameter>
                </boundary_condition>
            </boundary_conditions>
            
        </process_variable>
    </process_variables>

    <nonlinear_solvers>
        <nonlinear_solver>
            <name>basic_picard</name>
            <type>Picard</type>
            <max_iter>50</max_iter>
            <linear_solver>linear_solver</linear_solver>
        </nonlinear_solver>
    </nonlinear_solvers>

    <linear_solvers>
        <linear_solver>
            <name>linear_solver</name>
            <eigen>
                <solver_type>CG</solver_type>
                <precon_type>DIAGONAL</precon_type>
                <max_iteration_step>10000</max_iteration_step>
                <error_tolerance>1e-12</error_tolerance>
            </eigen>
        </linear_solver>
    </linear_solvers>

</OpenGeoSysProject>
"""

prj_path = input_dir / "simple_flow.prj"
prj_path.write_text(prj_content)

print("Project file created:", prj_path)


Project file created: E:\ADATA\LITHIUM\OGS\OGS_files\input\simple_flow.prj


In [303]:
print("PRJ exists:", (input_dir / "simple_flow.prj").exists())
print("Mesh exists:", (input_dir / "mesh5.vtu").exists())
print("GML exists:", (input_dir / "simple_geometry.gml").exists())

PRJ exists: True
Mesh exists: True
GML exists: True


In [304]:
import meshio

mesh = meshio.read(r"E:\ADATA\LITHIUM\OGS\OGS_files\input\mesh5.vtu")

print(mesh.cell_data_dict.keys())

dict_keys(['gmsh:physical', 'gmsh:geometrical'])


In [305]:
print(mesh.cell_data_dict)

{'gmsh:physical': {'triangle': array([301, 301, 301, ..., 306, 306, 306], shape=(12730,)), 'tetra': array([102, 102, 102, ..., 202, 202, 202], shape=(153274,))}, 'gmsh:geometrical': {'triangle': array([ 1,  1,  1, ..., 40, 40, 40], shape=(12730,)), 'tetra': array([ 4,  4,  4, ..., 14, 14, 14], shape=(153274,))}}


In [306]:
from pathlib import Path

ogs_bin = Path(r"C:\Users\dominic.becerra\Documents\OGS\ogs\bin")

ogs_root = ogs_bin.parent  # go one level up

print("OGS root:", ogs_root)

matches = []

for prj in ogs_root.rglob("*.prj"):
    try:
        text = prj.read_text(errors="ignore")
        if "<meshes>" in text:
            matches.append(prj)
    except:
        pass

print("Found:", len(matches))
for m in matches[:20]:
    print(m)

OGS root: C:\Users\dominic.becerra\Documents\OGS\ogs
Found: 0


In [307]:
from pathlib import Path

ogs_root = Path(r"C:\Users\dominic.becerra\Documents\OGS\ogs")

files = []
for pattern in ["*.xsd", "*.xml", "*.md", "*.txt"]:
    files.extend(ogs_root.rglob(pattern))

print("Found:", len(files))
for f in files[:50]:
    print(f)

Found: 8
C:\Users\dominic.becerra\Documents\OGS\ogs\bin\OpenGeoSysCND.xsd
C:\Users\dominic.becerra\Documents\OGS\ogs\bin\OpenGeoSysGLI.xsd
C:\Users\dominic.becerra\Documents\OGS\ogs\bin\OpenGeoSysNum.xsd
C:\Users\dominic.becerra\Documents\OGS\ogs\bin\OpenGeoSysProject.xsd
C:\Users\dominic.becerra\Documents\OGS\ogs\bin\OpenGeoSysSTN.xsd
C:\Users\dominic.becerra\Documents\OGS\ogs\README.txt
C:\Users\dominic.becerra\Documents\OGS\ogs\share\info\CMakeCache.txt
C:\Users\dominic.becerra\Documents\OGS\ogs\share\info\third_party_licenses.txt


In [308]:
from pathlib import Path

xsd_path = Path(r"C:\Users\dominic.becerra\Documents\OGS\ogs\bin\OpenGeoSysProject.xsd")

text = xsd_path.read_text(errors="ignore")

for keyword in ["meshes", "mesh"]:
    print("\n--- Searching:", keyword, "---")
    index = text.find(keyword)
    print("Found at:", index)
    if index != -1:
        print(text[index-500:index+1000])


--- Searching: meshes ---
Found at: 3570

      <xs:element name="boundary_conditions" type="bcListType" minOccurs="0" maxOccurs="1" />
      <xs:element name="source_terms" type="stListType" minOccurs="0" maxOccurs="1" />
    </xs:sequence>
  </xs:complexType>

  <!-- definition of file content -->
  <xs:element name="OpenGeoSysProject">
    <xs:complexType>
      <xs:sequence>
        <xs:element name="mesh" type="meshType" minOccurs="0"/>
        <xs:element name="geometry" type="xs:string" minOccurs="0"/>
        <xs:element name="meshes" minOccurs="0" maxOccurs="1">
          <xs:complexType>
            <xs:sequence>
              <xs:element name="mesh" type="meshType" minOccurs="0" maxOccurs="unbounded"/>
            </xs:sequence>
          </xs:complexType>
        </xs:element>
        <xs:element name="processes" minOccurs="0"/> <!--ignore-->
        <xs:element name="media" minOccurs="1" maxOccurs="1"/> <!--ignore-->
        <xs:element name="time_loop" minOccurs="0"/> <!

In [309]:
from pathlib import Path

prj_path = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\input\simple_flow.prj")

text = prj_path.read_text()

print(text[:1000])

<?xml version="1.0" encoding="ISO-8859-1"?>
<OpenGeoSysProject>

    <geometry>simple_geometry.gml</geometry>

    <meshes>
        <mesh>mesh5.vtu</mesh>
        <mesh>well1_completion.vtu</mesh>
        <mesh>well2_completion.vtu</mesh>
    </meshes>


    <processes>
        <process>
            <name>LiquidFlow</name>
            <type>LIQUID_FLOW</type>
            <integration_order>2</integration_order>
            <specific_body_force>0 0 0</specific_body_force>
            <process_variables>
                <process_variable>pressure</process_variable>
            </process_variables>
        </process>
    </processes>


    <media>
        <medium id="0">
            <phases>
                <phase>
                    <type>AqueousLiquid</type>
                    <properties>
                        <property>
                            <name>viscosity</name>
                            <type>Constant</type>
                            <value>1e-3</value>
              

In [310]:
from pathlib import Path

prj_path = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\input\simple_flow.prj")

text = prj_path.read_text()

for i, line in enumerate(text.splitlines(), start=1):
    if "<mesh>" in line or "</mesh>" in line:
        print(i, line)

7         <mesh>mesh5.vtu</mesh>
8         <mesh>well1_completion.vtu</mesh>
9         <mesh>well2_completion.vtu</mesh>
145                     <mesh>well1_completion</mesh>
150                     <mesh>well2_completion</mesh>


In [311]:
from pathlib import Path

prj_path = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\input\simple_flow.prj")
text = prj_path.read_text()

for i, line in enumerate(text.splitlines(), start=1):
    if "<mesh>" in line or "<geometry>" in line:
        print(i, line)

4     <geometry>simple_geometry.gml</geometry>
7         <mesh>mesh5.vtu</mesh>
8         <mesh>well1_completion.vtu</mesh>
9         <mesh>well2_completion.vtu</mesh>
145                     <mesh>well1_completion</mesh>
150                     <mesh>well2_completion</mesh>


In [312]:
from lxml import etree
from pathlib import Path

prj_path = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\input\simple_flow.prj")
xsd_path = Path(r"C:\Users\dominic.becerra\Documents\OGS\ogs\bin\OpenGeoSysProject.xsd")

schema = etree.XMLSchema(etree.parse(str(xsd_path)))
doc = etree.parse(str(prj_path))

print("Valid XML against XSD:", schema.validate(doc))

for error in schema.error_log:
    print(error.message)

Valid XML against XSD: True


In [313]:
from pathlib import Path

prj_path = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\input\simple_flow.prj")
text = prj_path.read_text()

for i, line in enumerate(text.splitlines(), start=1):
    if "<boundary_condition>" in line or "<type>" in line or "<mesh>" in line or "<parameter>" in line:
        print(i, line)

7         <mesh>mesh5.vtu</mesh>
8         <mesh>well1_completion.vtu</mesh>
9         <mesh>well2_completion.vtu</mesh>
16             <type>LIQUID_FLOW</type>
30                     <type>AqueousLiquid</type>
34                             <type>Constant</type>
39                             <type>Constant</type>
49                     <type>Constant</type>
54                     <type>Constant</type>
59                     <type>Constant</type>
64                     <type>Constant</type>
76                     <type>BackwardEuler</type>
79                     <type>DeltaX</type>
84                     <type>FixedTimeStepping</type>
98             <type>VTK</type>
113         <parameter>
115             <type>Constant</type>
118         <parameter>
120             <type>Constant</type>
123         <parameter>
125             <type>Constant</type>
128         <parameter>
130             <type>Constant</type>
143                 <boundary_condition>
144                     <type>Neuma

In [314]:
from pathlib import Path

prj_path = Path(r"E:\ADATA\LITHIUM\OGS\OGS_files\input\simple_flow.prj")
text = prj_path.read_text()

for i, line in enumerate(text.splitlines(), start=1):
    if "<mesh>" in line or "<meshes>" in line or "</meshes>" in line:
        print(i, line)

6     <meshes>
7         <mesh>mesh5.vtu</mesh>
8         <mesh>well1_completion.vtu</mesh>
9         <mesh>well2_completion.vtu</mesh>
10     </meshes>
145                     <mesh>well1_completion</mesh>
150                     <mesh>well2_completion</mesh>
